# PoC académico de recuperación multimodal de momentos con LanguageBind

Este notebook documenta una prueba de concepto para localizar momentos relevantes dentro de un vídeo de gameplay de **Helldivers 2** mediante **Zero-Shot Retrieval** y una adaptación ligera del espacio visual. La idea central es transformar un vídeo largo en una colección de ventanas temporales solapadas, representar cada ventana con embeddings de vídeo y audio, y comparar esas representaciones contra consultas en lenguaje natural.

Desde el punto de vista arquitectónico, el notebook no se limita a ejecutar scripts: organiza un pipeline completo de datos y modelos. Primero se construyen etiquetas débiles combinando transcripción automática, OCR del HUD y reglas visuales específicas del dominio. Después esas etiquetas se limpian, se fusionan y se enriquecen con un LLM para reducir la distancia semántica entre señales crudas y descripciones entrenables. Finalmente, se entrena un **Linear Probe** sobre embeddings congelados de LanguageBind y se usa para mejorar la búsqueda de eventos visuales caóticos con un coste de VRAM muy inferior al de un ajuste completo del modelo.

El resultado esperado es una guía reproducible para evaluar una hipótesis de tesis: si un modelo multimodal preentrenado puede adaptarse con un dataset pequeño, barato y semiautomático para recuperar escenas complejas que mezclan eventos visuales, HUD, audio de jugadores y contexto emocional.

## Imports y rutas

Esta sección prepara el entorno de ejecución del notebook y establece una frontera clara entre código experimental y código reutilizable. En lugar de copiar lógica dentro de las celdas, se importan funciones desde `scripts/`, lo que permite tratar el notebook como una capa de orquestación y documentación.

Aspectos importantes de esta celda:

- Se infiere `REPO_ROOT` para que el notebook funcione tanto si se abre desde la raíz del repositorio como desde `notebooks/`.
- Se añaden los módulos locales al `sys.path`, evitando instalaciones empaquetadas innecesarias durante la fase de prototipado.
- Se invalidan módulos ya cargados para que Jupyter recoja cambios recientes en los scripts sin reiniciar el kernel. Este detalle es relevante en investigación aplicada, donde el código auxiliar suele evolucionar mientras se documenta el experimento.
- Se importan de forma explícita las piezas del pipeline: carga de modelos, creación de ventanas, extracción de embeddings, fusión multimodal, ranking y exportación de clips.

En términos de **MLOps**, esta separación ayuda a mantener reproducibilidad: el notebook describe y ejecuta el flujo, mientras que las funciones versionables contienen la implementación.

In [1]:
from pathlib import Path
import importlib
import sys
import subprocess

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data" / "proxy_720p30.mp4").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

# Jupyter mantiene módulos importados entre re-ejecuciones. Los quitamos
# para que el notebook recoja cambios en scripts/tfvtg_poc.py al momento.
sys.modules.pop("scripts.tfvtg_poc", None)
sys.modules.pop("scripts.build_languagebind_index", None)
importlib.invalidate_caches()

from scripts.tfvtg_poc import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_AUDIO_WEIGHT,
    DEFAULT_CLIP_CONTEXT_SECONDS,
    DEFAULT_NUM_WORKERS,
    DEFAULT_STRIDE_SECONDS,
    DEFAULT_WINDOW_SECONDS,
    DEFAULT_VIDEO_WEIGHT,
    export_result_clips,
    extract_scene_embeddings_batched,
    extract_text_embedding,
    load_audio_track,
    load_languagebind_models,
    make_windows,
    open_video_reader,
    print_results,
    rank_windows,
    weighted_fuse_embeddings,
    require_cuda,
)
from scripts.build_languagebind_index import (
    build_languagebind_index,
    default_index_dir,
    load_languagebind_index,
)


## Configuración experimental

Esta celda define los hiperparámetros globales del experimento. Aunque varios valores tienen defaults centralizados en los scripts, aquí se fijan de forma explícita para que el evaluador pueda interpretar el diseño experimental sin inspeccionar todo el repositorio.

#### Ventanas temporales y densidad de muestreo

- `WINDOW_SECONDS = 15` usa ventanas largas para capturar el ciclo completo de un evento de gameplay: preparación, explosión o fallo, reacción verbal y posible actualización del HUD. En Helldivers 2, muchos momentos memorables no son instantáneos, sino secuencias de varios segundos.
- `STRIDE_SECONDS = 5` introduce solapamiento entre ventanas. Esto reduce el riesgo de cortar un evento justo en el borde de una ventana y permite que la misma escena sea observada desde distintos alineamientos temporales.
- `CLIP_CONTEXT_SECONDS = 2` añade margen al exportar resultados, útil para validación humana porque permite ver el antecedente y la consecuencia inmediata del momento recuperado.

#### Embeddings, índice y rendimiento

- `EMBEDDING_MODE = "load"` prioriza reproducibilidad y rapidez: los embeddings ya calculados se leen desde disco y las consultas pueden repetirse sin recodificar el vídeo.
- `BATCH_SIZE = 48` busca alta ocupación de GPU durante la extracción de embeddings. Si aparece **Out Of Memory**, este es el primer parámetro que debe reducirse.
- `NUM_WORKERS = 4` permite preprocesar vídeo y audio en paralelo mientras la GPU consume lotes, mejorando el throughput del pipeline.
- `TOP_K = 3` produce una lista corta de candidatos para inspección humana; en una PoC académica es preferible validar pocos resultados con atención antes que exportar decenas de clips ruidosos.

#### Pesos multimodales

- `VIDEO_WEIGHT` y `AUDIO_WEIGHT` se mantienen como configuración base equilibrada para construir embeddings de escena. Esta decisión refleja la premisa inicial de LanguageBind: vídeo y audio están alineados en un espacio común centrado en texto.
- En la consulta final, cuando existe el adapter de Helldivers, se usa una **Late Fusion** más visual (`0.8` vídeo, `0.2` audio). La justificación es que el adapter solo transforma el espacio visual; si se mezclara al 50/50 con audio genérico, se diluiría parte de la especialización aprendida. El peso `0.8` preserva la adaptación visual sin ignorar completamente risas, gritos o explosiones.

#### Entrenamiento del adapter

- `ADAPTER_EPOCHS = 300` puede parecer alto, pero el entrenamiento no ajusta LanguageBind completo: solo una capa lineal pequeña sobre embeddings congelados. Al tener pocos parámetros y un dataset reducido, hacen falta muchas pasadas para que el **Linear Probe** converja de forma estable.
- `ADAPTER_BATCH_SIZE = 4` se usa solo durante la fase de precomputación de embeddings congelados, donde todavía se decodifican ventanas de vídeo y se ejecuta el backbone. Es una decisión conservadora para evitar saturar VRAM.
- `ADAPTER_LEARNING_RATE = 5e-3` es agresivo en comparación con fine-tuning profundo, pero razonable para una cabeza lineal pequeña entrenada con AdamW. El backbone permanece congelado, por lo que el riesgo de destruir representaciones generales es mucho menor.

#### Etiquetado automático

- `AUTO_LABEL_KEEP_TOP_PCT = 0.15` conserva solo el 15% de ventanas con mayor energía RMS antes de llamar a Whisper. Esto reduce coste computacional y concentra transcripción en momentos con habla, risas, gritos o explosiones.
- `OCR_MIN_CONFIDENCE = 0.35` acepta detecciones OCR relativamente débiles porque el HUD aparece con movimiento, efectos visuales y compresión. El ruido se corrige después con limpieza, fusión y reescritura semántica.
- `REWRITE_MODEL = "llama3.1"` se usa localmente para convertir etiquetas crudas en descripciones ricas en inglés, que son más compatibles con el espacio textual preentrenado de LanguageBind.

In [34]:
VIDEO_PATH = REPO_ROOT / "data" / "proxy_720p30.mp4"
CACHE_DIR = REPO_ROOT / "cache_dir"
CLIP_OUTPUT_DIR = REPO_ROOT / "outputs" / "validation_clips"

WINDOW_SECONDS = 15 #DEFAULT_WINDOW_SECONDS  # 8.0
STRIDE_SECONDS = 5 #DEFAULT_STRIDE_SECONDS  # 2.0
INDEX_DIR = REPO_ROOT / default_index_dir(VIDEO_PATH, WINDOW_SECONDS, STRIDE_SECONDS)

CLIP_CONTEXT_SECONDS = DEFAULT_CLIP_CONTEXT_SECONDS  # 2.0

# Opciones: "load", "build_if_missing", "compute_session".
EMBEDDING_MODE = "load"

BATCH_SIZE = DEFAULT_BATCH_SIZE  # 48
NUM_WORKERS = DEFAULT_NUM_WORKERS  # 4
TOP_K = 3

VIDEO_WEIGHT = DEFAULT_VIDEO_WEIGHT  # 0.5
AUDIO_WEIGHT = DEFAULT_AUDIO_WEIGHT  # 0.5

USE_HELLDIVERS_ADAPTER = True  # True
HELLDIVERS_ADAPTER_PATH = REPO_ROOT / "data/models/helldivers_adapter.pth"
ADAPTER_EPOCHS = 300  # Necesitamos muchos
ADAPTER_BATCH_SIZE = 4  
ADAPTER_NUM_WORKERS = 4  
ADAPTER_LEARNING_RATE = 5e-3 # y que sea agresivo

WHISPER_DATA_DIR = REPO_ROOT / "data" / "whisper"
WHISPER_DATA_DIR.mkdir(parents=True, exist_ok=True)

AUTO_LABEL_OUTPUT = WHISPER_DATA_DIR / "dataset_whisper.json"
AUTO_LABEL_WHISPER_MODEL = "openai/whisper-large-v3-turbo"  # openai/whisper-large-v3-turbo
AUTO_LABEL_KEEP_TOP_PCT = 0.15  # 0.15

CLEAN_DATASET_OUTPUT = WHISPER_DATA_DIR / "dataset_cleaned.json"

OCR_DATA_DIR = REPO_ROOT / "data" / "ocr"
OCR_DATA_DIR.mkdir(parents=True, exist_ok=True)

OCR_DATASET_OUTPUT = OCR_DATA_DIR / "dataset_ocr_raw.json"
OCR_WINDOW_SECONDS = 15.0  # 15.0
OCR_STRIDE_SECONDS = 5.0  # 5.0
OCR_MIN_CONFIDENCE = 0.35  # 0.35

OCR_CLEAN_DATASET_OUTPUT = OCR_DATA_DIR / "dataset_ocr_cleaned.json"
MASTER_DATASET_OUTPUT = REPO_ROOT / "data" / "master_dataset_raw.json"

MASTER_DATASET_FINAL_OUTPUT = REPO_ROOT / "data" / "master_dataset_final.json"
REWRITE_MODEL = "llama3.1" 

print(f"Repo root: {REPO_ROOT}")
print(f"Video: {VIDEO_PATH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Video/audio weights: {VIDEO_WEIGHT}/{AUDIO_WEIGHT}")
print(f"Use Helldivers adapter: {USE_HELLDIVERS_ADAPTER}")
print(f"Index dir: {INDEX_DIR}")
print(f"Embedding mode: {EMBEDDING_MODE}")
print(f"Window/stride: {WINDOW_SECONDS}s / {STRIDE_SECONDS}s")
print(f"Validation clip context: +/-{CLIP_CONTEXT_SECONDS}s")


Repo root: /home/ruben/Documents/AINE/aine-highlights
Video: /home/ruben/Documents/AINE/aine-highlights/data/proxy_720p30.mp4
Batch size: 48
DataLoader workers: 4
Video/audio weights: 0.5/0.5
Use Helldivers adapter: True
Index dir: /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5
Embedding mode: load
Window/stride: 15s / 5s
Validation clip context: +/-2.0s


## Dataset Whisper: etiquetado débil desde audio

Esta celda ejecuta `auto_label_dataset.py`, que construye un primer dataset automático a partir del canal de audio. La razón arquitectónica de este paso es aprovechar una señal barata y abundante: las reacciones humanas. En vídeos de gameplay cooperativo, los gritos, risas y comentarios suelen coincidir con momentos visualmente relevantes, incluso cuando el evento exacto no está escrito en pantalla.

Internamente, el script aplica una cascada en dos fases:

1. **Sliding Window**: el vídeo se divide en ventanas temporales solapadas usando `WINDOW_SECONDS` y `STRIDE_SECONDS`. Cada ventana se convierte en una unidad candidata de entrenamiento con `start_s`, `end_s` y `window_index`.
2. **Filtrado por energía RMS**: antes de invocar Whisper, el script calcula la energía RMS del audio de cada ventana. Esta métrica aproxima la intensidad acústica y permite conservar solo las ventanas más activas (`AUTO_LABEL_KEEP_TOP_PCT = 0.15`). Es una optimización importante: transcribir todo el vídeo sería más caro y produciría muchas etiquetas vacías o irrelevantes.
3. **Transcripción con Whisper**: las ventanas supervivientes se remuestrean a 16 kHz mono y se procesan con `openai/whisper-large-v3-turbo`. La llamada fuerza el idioma a español y utiliza un prompt contextual que informa a Whisper de que se trata de audio informal de jugadores andaluces en Helldivers 2. Esto ayuda a reducir errores en acentos, jerga, risas, explosiones y habla superpuesta.
4. **Control mínimo de calidad**: se descartan tokens de ruido y transcripciones demasiado cortas, dejando solo textos con suficiente contenido para servir como etiqueta débil.

El resultado no pretende ser un dataset perfecto. Su valor es producir rápidamente pares ventana-texto que capturan la dimensión emocional y conversacional del gameplay, una señal que el OCR por sí solo no puede observar.

In [3]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "auto_label_dataset.py"),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(AUTO_LABEL_OUTPUT),
        "--window-seconds",
        str(WINDOW_SECONDS),
        "--stride-seconds",
        str(STRIDE_SECONDS),
        "--keep-top-pct",
        str(AUTO_LABEL_KEEP_TOP_PCT),
        "--whisper-model",
        AUTO_LABEL_WHISPER_MODEL,
        "--language",
        "spanish",
    ],
    check=True,
)

print(f"Dataset Whisper guardado en: {AUTO_LABEL_OUTPUT}")

Loaded video=/home/ruben/Documents/AINE/aine-highlights/data/proxy_720p30.mp4, duration=6221.20s, fps=30.00, windows=1243, audio_sr=48000
Phase 1 kept 186/1243 windows (top 15%).


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Whisper labeling: 100%|██████████| 186/186 [01:24<00:00,  2.21it/s]


Saved 183 labels to /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_whisper.json
Dataset Whisper guardado en: /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_whisper.json


### Limpieza del dataset Whisper

Esta celda ejecuta `clean_dataset.py` sobre las transcripciones de Whisper. El objetivo no es corregir lingüísticamente el texto, sino eliminar duplicados causados por el solapamiento de ventanas. Sin esta limpieza, un mismo grito o comentario podría aparecer varias veces y sesgar el entrenamiento hacia momentos repetidos.

El script implementa una forma ligera de **Non-Maximum Suppression (NMS)** temporal aplicada a texto:

- Ordena las ventanas por tiempo de inicio.
- Agrupa ventanas consecutivas si están suficientemente cerca (`max_start_gap_seconds`) y si sus textos comparten vocabulario relevante.
- Calcula **similitud de Jaccard** entre conjuntos de palabras de contenido, ignorando términos muy cortos que aportan poco significado.
- Dentro de cada grupo, conserva el "champion": la transcripción más rica, medida por número de palabras y longitud total.

La analogía con NMS en visión por computador es directa: varias detecciones solapadas compiten por representar el mismo evento, y se conserva una sola instancia de mayor calidad. Aquí la caja espacial se reemplaza por proximidad temporal y la puntuación visual por riqueza textual.

In [4]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "clean_dataset.py"),
        "--input",
        str(AUTO_LABEL_OUTPUT),
        "--output",
        str(CLEAN_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset Whisper limpio guardado en: {CLEAN_DATASET_OUTPUT}")


Original windows: 183 -> Cleaned windows: 105
Saved cleaned dataset to /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_cleaned.json
Dataset Whisper limpio guardado en: /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_cleaned.json


## Dataset OCR: eventos explícitos del HUD

Esta celda ejecuta `auto_label_ocr.py`, que etiqueta ventanas a partir de información escrita en pantalla. La razón de incluir OCR es complementar Whisper: el audio captura reacciones humanas, pero el HUD ofrece señales estructuradas sobre el estado del juego, como eliminación, refuerzos, brechas de enemigos o uso de estratagemas.

El script combina OCR, reglas geométricas y visión clásica:

- **Muestreo a 1 FPS**: en lugar de analizar todos los frames, toma aproximadamente un frame por segundo dentro de cada ventana. Esto reduce drásticamente el coste y sigue siendo suficiente para eventos de HUD que permanecen visibles durante varios frames.
- **Regiones de Interés (ROIs)**: con `--roi hud`, el análisis se restringe a zonas donde Helldivers 2 suele mostrar información útil: banner central, cabecera superior, bloque superior izquierdo y panel inferior izquierdo. Esta restricción mejora precisión y velocidad frente a OCR de pantalla completa.
- **Preprocesado del HUD**: se genera una máscara de alto contraste que refuerza texto blanco y rojo, colores frecuentes en avisos del juego, y se aplica una pequeña dilatación para reconectar glifos finos tras el umbralizado.
- **Filtro de menús**: si se detectan palabras de la nave o menús (`GESTIÓN`, `NAVE`, `ADQUISICIONES`, `ARMERÍA`, `DESTRUCTOR`), se etiqueta `EN LA NAVE` y se evita mezclar ese estado con eventos de combate. Este filtro reduce falsos positivos de estratagemas o elementos de interfaz que aparecen fuera de misión.
- **Detección de estratagemas y estado de escuadra**: el OCR busca términos como `ORBITAL`, `ÁGUILA`, `BOMBA` o `ATAQUE` en la zona superior izquierda, y señales de disponibilidad en el panel inferior izquierdo para inferir `COMPAÑERO CAÍDO`.
- **Detección de descenso en cápsula sin deep learning**: para `DESCENSO EN CÁPSULA`, el script no entrena un detector. Usa espacio de color **HSV** en una ROI centro-inferior y busca una proporción suficiente de píxeles naranja/amarillo saturados o blancos muy brillantes, patrones característicos de las llamas del Hellpod. Esta decisión es deliberadamente eficiente: resuelve un caso visual específico con visión clásica y evita añadir otro modelo.

El dataset OCR aporta etiquetas más discretas y semánticamente estables que el audio. Por eso es especialmente útil para eventos de juego que un jugador puede no verbalizar, pero que aparecen de forma explícita en pantalla.

In [6]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "auto_label_ocr.py"),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(OCR_DATASET_OUTPUT),
        "--window-seconds",
        str(OCR_WINDOW_SECONDS),
        "--stride-seconds",
        str(OCR_STRIDE_SECONDS),
        "--roi",
        "hud",
        "--min-confidence",
        str(OCR_MIN_CONFIDENCE),
    ],
    check=True,
)

print(f"Dataset OCR guardado en: {OCR_DATASET_OUTPUT}")


OCR labeling: 100%|██████████| 1243/1243 [00:00<00:00, 314313.61it/s]


Saved 301 OCR labels to /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_raw.json
Dataset OCR guardado en: /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_raw.json


### Limpieza del dataset OCR

Esta celda reutiliza `clean_dataset.py` para deduplicar las etiquetas OCR. Aunque las etiquetas del HUD suelen ser más cortas que las transcripciones, el solapamiento temporal también puede duplicar un mismo evento en varias ventanas consecutivas.

La limpieza vuelve a aplicar **NMS temporal basada en texto**:

- Ventanas próximas se consideran candidatas a pertenecer al mismo evento.
- La **similitud de Jaccard** detecta si comparten etiquetas como `ELIMINADO`, `REFUERZOS LISTOS` o `BRECHA DE BICHOS`.
- Se conserva la variante más informativa cuando varias ventanas describen el mismo suceso.

Este paso es importante antes de fusionar modalidades. Si el OCR entrara duplicado en el dataset maestro, podría dominar artificialmente al audio y aumentar el riesgo de **Overfitting** del adapter hacia unas pocas etiquetas repetidas del HUD.

In [7]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "clean_dataset.py"),
        "--input",
        str(OCR_DATASET_OUTPUT),
        "--output",
        str(OCR_CLEAN_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset OCR limpio guardado en: {OCR_CLEAN_DATASET_OUTPUT}")


Original windows: 301 -> Cleaned windows: 69
Saved cleaned dataset to /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_cleaned.json
Dataset OCR limpio guardado en: /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_cleaned.json


## Unión de datasets: audio, HUD y efecto eco

Esta celda ejecuta `merge_datasets.py`, que combina el dataset limpio de Whisper con el dataset limpio de OCR. El propósito arquitectónico es construir una etiqueta multimodal más completa que cualquiera de las dos fuentes por separado: el audio aporta reacción humana y el HUD aporta estado objetivo del juego.

El script utiliza tres estrategias:

- Si una ventana de Whisper y una de OCR corresponden al mismo `window_index`, se fusionan directamente.
- Si no coinciden exactamente, se permite una asociación temporal asimétrica: el audio puede ocurrir desde 2 segundos antes hasta 8 segundos después del evento visual del HUD. Este diseño implementa el **Asymmetrical Echo Effect**: en gameplay real, la consecuencia emocional suele llegar después del estímulo visual. Por ejemplo, una eliminación aparece en pantalla y varios segundos después llega la queja, la risa o el grito del jugador.
- Si una instancia no encuentra pareja, se conserva como `Audio-only` u `OCR-only` en lugar de descartarse. Esto evita perder eventos silenciosos o reacciones sin texto de HUD.

El resultado usa marcadores explícitos (`[Audio: ...]` y `[HUD: ...]`) para que la siguiente etapa pueda distinguir el origen de cada señal. Esta estructura intermedia mantiene trazabilidad antes de pasar a una descripción natural más rica.

In [8]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "merge_datasets.py"),
        "--whisper",
        str(CLEAN_DATASET_OUTPUT),
        "--ocr",
        str(OCR_CLEAN_DATASET_OUTPUT),
        "--output",
        str(MASTER_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset maestro guardado en: {MASTER_DATASET_OUTPUT}")


Whisper: 105 + OCR: 69 -> Master: 165
Saved master dataset to /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_raw.json
Dataset maestro guardado en: /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_raw.json


## Reescritura semántica del dataset maestro

Esta celda ejecuta `rewrite_dataset.py`, que transforma etiquetas crudas en descripciones inglesas objetivas y entrenables. La motivación es resolver el **Semantic Gap** entre lo que tenemos como datos débiles y lo que LanguageBind espera comparar en su espacio texto-vídeo.

Las etiquetas crudas suelen mezclar fragmentos como:

- Transcripciones en español, con jerga, acentos, risas o frases incompletas.
- Etiquetas de HUD en formato corto, por ejemplo `ELIMINADO` o `BRECHA DE BICHOS`.
- Eventos implícitos donde la emoción o la causalidad no aparecen literalmente.

`rewrite_dataset.py` usa `llama3.1` mediante Ollama con un prompt de sistema especializado en anotación de Helldivers 2. El modelo no debe traducir de forma literal; debe inferir situación, emoción y causalidad. Por ejemplo, una frase de frustración junto a `ELIMINADO` puede convertirse en una descripción como una muerte accidental durante extracción, y una pregunta nerviosa junto a `BRECHA DE BICHOS` puede convertirse en una escena de pánico ante una invasión enemiga.

Esta reescritura genera descripciones en inglés porque el espacio textual de muchos modelos multimodales preentrenados, incluido LanguageBind, suele estar más densamente alineado con captions inglesas. En otras palabras, el LLM actúa como puente semántico entre señales locales y ruidosas y una formulación compatible con **Contrastive Learning**.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "rewrite_dataset.py"),
        "--input",
        str(MASTER_DATASET_OUTPUT),
        "--output",
        str(MASTER_DATASET_FINAL_OUTPUT),
        "--model",
        REWRITE_MODEL,
    ],
    check=True,
)

print(f"Dataset maestro final guardado en: {MASTER_DATASET_FINAL_OUTPUT}")


Rewriting labels:  99%|█████████▉| 164/165 [01:07<00:00,  4.42it/s]

Saved final dataset to /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_final.json
Dataset maestro final guardado en: /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_final.json


Rewriting labels: 100%|██████████| 165/165 [01:07<00:00,  2.44it/s]


## Entrenamiento del adapter Helldivers

Esta celda ejecuta `train_linear_probe.py`, que entrena un **Linear Probe** sobre embeddings de vídeo congelados de LanguageBind. La decisión clave es adaptar una proyección pequeña, no hacer **Full Fine-Tuning** del backbone. Esto reduce masivamente el consumo de VRAM, disminuye el riesgo de sobreajuste catastrófico y permite trabajar con un dataset pequeño generado de forma semiautomática.

Internamente, el script sigue este flujo:

1. Carga el dataset final y decodifica las ventanas de vídeo asociadas.
2. Tokeniza las descripciones enriquecidas generadas por `llama3.1`.
3. Ejecuta LanguageBind en modo congelado para precomputar embeddings de vídeo y texto. Tras esta fase, el backbone se libera de memoria.
4. Entrena `VideoTextAdapter`, una capa lineal con `dropout`, sobre los embeddings de vídeo.
5. Optimiza una pérdida contrastiva simétrica tipo **InfoNCE**, donde cada vídeo debe acercarse a su texto emparejado y alejarse del resto de textos del lote.

#### Por qué se usa Full Batch

El script configura el lote de entrenamiento como el dataset completo una vez precomputados los embeddings. En **Contrastive Learning**, los ejemplos negativos son los demás elementos del batch. Por tanto, un batch pequeño ofrece pocos negativos y produce una señal de entrenamiento débil. El **Full Batch** maximiza la matriz vídeo-texto y hace que cada paso compare cada escena contra todas las demás descripciones disponibles.

#### Justificación de hiperparámetros

- `ADAPTER_EPOCHS = 300`: al entrenar solo una capa lineal, cada época es barata, pero la convergencia puede requerir muchas pasadas porque el dataset es pequeño y las diferencias semánticas son sutiles.
- `ADAPTER_LEARNING_RATE = 5e-3`: una tasa alta sería peligrosa para un backbone completo, pero es adecuada para una proyección lineal con pocos parámetros y embeddings congelados.
- `ADAPTER_BATCH_SIZE = 4`: se aplica en la fase de precomputación con vídeo real, que es la parte intensiva en VRAM. Después, el entrenamiento contrastivo opera sobre embeddings ya calculados y puede usar Full Batch.
- `dropout = 0.1` y `weight_decay = 1e-2` actúan como regularización ligera frente a **Overfitting**, especialmente importante porque las etiquetas proceden de un proceso automático y pueden contener ruido.

El fichero final `data/models/helldivers_adapter.pth` contiene solo los pesos del adapter. Esto lo hace fácil de versionar, cargar y comparar contra la línea base zero-shot.

In [35]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "train_linear_probe.py"),
        "--dataset",
        str(MASTER_DATASET_FINAL_OUTPUT),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(HELLDIVERS_ADAPTER_PATH),
        "--cache-dir",
        str(CACHE_DIR),
        "--epochs",
        str(ADAPTER_EPOCHS),
        "--batch-size",
        str(ADAPTER_BATCH_SIZE),
        "--num-workers",
        str(ADAPTER_NUM_WORKERS),
        "--lr",
        str(ADAPTER_LEARNING_RATE),
    ],
    check=True,
)

print(f"Adapter Helldivers guardado en: {HELLDIVERS_ADAPTER_PATH}")


/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
Epoch 300/300: 100%|██████████| 1/1 [00:00<00:00, 381.75it/s, loss=0.7181]


Saved adapter weights to /home/ruben/Documents/AINE/aine-highlights/data/models/helldivers_adapter.pth
Adapter Helldivers guardado en: /home/ruben/Documents/AINE/aine-highlights/data/models/helldivers_adapter.pth


## GPU y carga de modelos

Estas celdas verifican que existe una GPU CUDA disponible y cargan las ramas de vídeo y audio de LanguageBind. La exigencia de CUDA no es un capricho técnico: el pipeline procesa muchas ventanas temporales, cada una con frames y audio transformados, y necesita inferencia batched para que la PoC sea práctica.

LanguageBind se usa porque alinea distintas modalidades en un espacio común compatible con texto. En este notebook se emplean dos ramas principales:

- **Vídeo**: representa la apariencia y dinámica visual de cada ventana.
- **Audio**: representa habla, explosiones, risas, gritos y ambiente sonoro.

También se carga el tokenizer textual asociado a la rama de vídeo, que será el encargado de transformar consultas en embeddings comparables. Los modelos se cargan una sola vez para evitar inicializaciones repetidas y para que las siguientes celdas puedan alternar entre construir índices, cargar embeddings o hacer consultas sin duplicar memoria innecesariamente.

In [36]:
import torch

device = require_cuda()
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"Using GPU: {gpu_name}")
print(f"Compute capability: {capability[0]}.{capability[1]}")
print(f"PyTorch: {torch.__version__}, CUDA runtime: {torch.version.cuda}")


Using GPU: NVIDIA GeForce RTX 5070 Ti
Compute capability: 12.0
PyTorch: 2.11.0+cu128, CUDA runtime: 12.8


In [37]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(
    video_model,
    audio_model,
    video_processor,
    audio_processor,
    text_tokenizer,
) = load_languagebind_models(cache_dir=CACHE_DIR, device=device)

print("Loaded LanguageBind video/audio models on CUDA.")


Loaded LanguageBind video/audio models on CUDA.


## Embeddings e índice de recuperación

Esta sección decide de dónde salen los embeddings de escena. Arquitectónicamente, separa el coste caro de codificar vídeo/audio del coste barato de consultar con texto. Esta separación es fundamental para iterar: una vez indexado el vídeo, se pueden probar muchas consultas sin volver a pasar todas las ventanas por LanguageBind.

Modos disponibles:

- `load`: carga un índice persistente desde `INDEX_DIR`. Es el modo preferido para análisis y escritura académica, porque garantiza que las consultas se evalúan sobre una representación fija.
- `build_if_missing`: construye el índice si no existe y lo guarda en disco. Es útil para reproducir el experimento desde cero.
- `compute_session`: calcula embeddings solo en memoria. Es flexible para pruebas rápidas, pero menos reproducible porque no deja un artefacto persistente.

Durante la construcción del índice, cada ventana temporal se codifica con vídeo y audio por separado. Después se aplica **Late Fusion** mediante una media ponderada de embeddings normalizados. Esta normalización es importante: evita que una modalidad domine simplemente por tener mayor norma vectorial. El índice guarda embeddings de vídeo, audio y escena fusionada, lo que permite diagnósticos posteriores y cambios de peso durante la consulta.

El uso de ventanas solapadas convierte un problema de búsqueda en vídeo largo en un problema de ranking sobre fragmentos comparables. Cada consulta textual se evaluará contra todas las ventanas mediante similitud coseno.

In [38]:
if EMBEDDING_MODE not in {"load", "build_if_missing", "compute_session"}:
    raise ValueError('EMBEDDING_MODE must be one of: "load", "build_if_missing", "compute_session"')

if EMBEDDING_MODE == "load":
    if not (INDEX_DIR / "metadata.json").exists():
        raise FileNotFoundError(
            f"No pregenerated index found at {INDEX_DIR}. "
            "Switch EMBEDDING_MODE to 'build_if_missing' or 'compute_session'."
        )
    index = load_languagebind_index(INDEX_DIR)
    windows = index.windows
    scene_embeddings = index.scene_embeddings.to(device=device, dtype=torch.float32)
    duration_s = float(index.metadata["duration_s"])
    print(f"Loaded pregenerated index: {INDEX_DIR}")

elif EMBEDDING_MODE == "build_if_missing":
    if (INDEX_DIR / "metadata.json").exists():
        index = load_languagebind_index(INDEX_DIR)
        print(f"Loaded existing index: {INDEX_DIR}")
    else:
        print(f"No existing index at {INDEX_DIR}; building and saving it now.")
        index = build_languagebind_index(
            video_path=VIDEO_PATH,
            output_dir=INDEX_DIR,
            window_seconds=WINDOW_SECONDS,
            stride_seconds=STRIDE_SECONDS,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            cache_dir=CACHE_DIR,
            dtype="float16",
            overwrite=False,
        )
    windows = index.windows
    scene_embeddings = index.scene_embeddings.to(device=device, dtype=torch.float32)
    duration_s = float(index.metadata["duration_s"])

else:  # EMBEDDING_MODE == "compute_session"
    print("Computing embeddings in memory for this notebook session only.")
    video_reader, fps, duration_s = open_video_reader(VIDEO_PATH)
    audio_track = load_audio_track(VIDEO_PATH)
    windows = make_windows(
        duration_s=duration_s,
        window_seconds=WINDOW_SECONDS,
        stride_seconds=STRIDE_SECONDS,
    )
    with torch.no_grad():
        scene_embeddings = extract_scene_embeddings_batched(
            windows=windows,
            video_path=VIDEO_PATH,
            video_reader=video_reader,
            fps=fps,
            audio_track=audio_track,
            video_model=video_model,
            audio_model=audio_model,
            video_processor=video_processor,
            audio_processor=audio_processor,
            device=device,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            video_weight=VIDEO_WEIGHT,
            audio_weight=AUDIO_WEIGHT,
        )

print(
    f"Embeddings ready: mode={EMBEDDING_MODE}, windows={len(windows)}, "
    f"scene_embeddings={tuple(scene_embeddings.shape)}, device={scene_embeddings.device}"
)
print(f"First 3 windows: {windows[:3]}")


Loaded pregenerated index: /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5
Embeddings ready: mode=load, windows=1243, scene_embeddings=(1243, 768), device=cuda:0
First 3 windows: [WindowSpec(start_s=0.0, end_s=15.0), WindowSpec(start_s=5.0, end_s=20.0), WindowSpec(start_s=10.0, end_s=25.0)]


## Consulta de texto y ranking multimodal

Esta sección realiza la búsqueda propiamente dicha. Una consulta en lenguaje natural se codifica como embedding textual y se compara contra las ventanas disponibles. El resultado es un ranking `Top-K` de momentos candidatos, ordenados por similitud coseno.

La celda incluye varias consultas de ejemplo porque la calidad de **Zero-Shot Retrieval** depende mucho de la formulación. Consultas descriptivas, causales y emocionalmente explícitas suelen funcionar mejor que palabras clave aisladas, ya que se parecen más a captions de entrenamiento multimodal.

Si `USE_HELLDIVERS_ADAPTER` está activo y existe `data/models/helldivers_adapter.pth`, se aplica el adapter al espacio visual antes de fusionar modalidades. Aquí aparece una decisión importante de inferencia:

- Se usa `video_weight = 0.8` y `audio_weight = 0.2` en la **Late Fusion** final.
- La razón es que el adapter fue entrenado únicamente sobre embeddings visuales; por tanto, el espacio visual queda especializado en eventos de Helldivers.
- El audio se mantiene como apoyo para capturar risas, gritos o explosiones, pero con menor peso para no diluir la adaptación aprendida con el espacio de audio genérico.

Este diseño permite comparar dos escenarios: la recuperación base de LanguageBind y la recuperación adaptada al dominio. Si el adapter mejora resultados para consultas como explosiones, estratagemas o muertes accidentales, se obtiene evidencia de que una adaptación lineal pequeña puede mover el espacio multimodal hacia el dominio de interés sin reentrenar el modelo completo.

In [46]:
import torch.nn.functional as F
from scripts.train_linear_probe import VideoTextAdapter

CONTEXT = "A gameplay clip of a first-person sci-fi video game where "
QUERY_1 = "the camera suddenly falls to the ground and the screen turns red while the gamer groans in defeat."
QUERY_2 = "a glowing beacon causes a huge explosion that accidentally blows up a teammate, while gamers laugh hysterically and one person shouts in anger."
QUERY_3 = "a massive bomb suddenly explodes sending a friendly soldier flying through the air, triggering loud laughter and an angry yell over voice chat."
QUERY_4 = "a player accidentally drops a giant airstrike on the team and says oops, followed by loud explosions, laughing, and angry screaming."
QUERY_5 = "reinforcements are available"
QUERY_6 = "a player is eliminated"
QUERY_7 = "a player uses a stratagem"
QUERY_8 = "A player is killed by friendly fire"
QUERY_9 = "A dropship lands to rescue the players"
QUERY_10 = "A barrage of airstrikes is called in, causing massive explosions and chaos on the battlefield"
QUERY_11 = "An objective is completed"
QUERY_12 = "An eagle airstrike is called in"
QUERY_13 = "a player panics and frantically asks for controls while being overwhelmed by a massive bug breach"
QUERY_14 = "a player expresses heavy frustration after being accidentally incinerated by the dropship engines during extraction"
QUERY_15 = "a player drops into the battlefield in a hellpod, returning to the chaotic action"
QUERY_16 = "the squad is in a critical situation with no reinforcements available"
QUERY_17 = "a player frantically runs away under heavy attack, creating extreme camera chaos"
QUERY_18 = "a teammate hilariously betrays the squad, resulting in an accidental death and loud laughter over voice chat"
QUERY_19 = "a player desperately calls in an orbital stratagem to stop a massive bug breach"
QUERY_20 = "a comrade is down in battle and the player realizes reinforcements are ready to be deployed"
QUERY_21 = "the squad completes a critical mission objective, bringing the operation closer to success"
QUERY_22 = "a player is eliminated in the middle of a massive firefight while teammates scream in panic"
QUERY_23 = "a player expresses shock and grief after realizing their own mistake led to a teammate's death"
QUERY_24 = "the squad successfully extracts, but not without chaotic screams and last-second friendly fire"

adapter = None
if USE_HELLDIVERS_ADAPTER and HELLDIVERS_ADAPTER_PATH.exists():
    embedding_dim = int(index.video_embeddings.shape[-1]) if "index" in globals() else int(scene_embeddings.shape[-1])
    adapter = VideoTextAdapter(embedding_dim=embedding_dim).to(device)
    adapter.load_state_dict(torch.load(HELLDIVERS_ADAPTER_PATH, map_location=device))
    adapter.eval()
    print(f"Adapter Helldivers cargado: {HELLDIVERS_ADAPTER_PATH}")
elif USE_HELLDIVERS_ADAPTER:
    print(f"Adapter Helldivers no encontrado, usando embeddings base: {HELLDIVERS_ADAPTER_PATH}")

with torch.no_grad():
    text_embedding = extract_text_embedding(
        model=video_model,
        tokenizer=text_tokenizer,
        query=QUERY_10,
        device=device,
    )
    if "index" in globals() and hasattr(index, "video_embeddings") and hasattr(index, "audio_embeddings"):
        video_embeddings = index.video_embeddings.to(device=device, dtype=torch.float32)
        audio_embeddings = index.audio_embeddings.to(device=device, dtype=torch.float32)
        if adapter is not None:
            video_embeddings = F.normalize(adapter(video_embeddings), dim=-1)
        query_scene_embeddings = weighted_fuse_embeddings(
            video_embeddings,
            audio_embeddings,
            video_weight= 0.8, #VIDEO_WEIGHT
            audio_weight=0.2, #AUDIO_WEIGHT
        )
    else:
        query_scene_embeddings = scene_embeddings

    results = rank_windows(
        windows=windows,
        scene_embeddings=query_scene_embeddings,
        text_embedding=text_embedding,
        top_k=TOP_K,
    )


Adapter Helldivers cargado: /home/ruben/Documents/AINE/aine-highlights/data/models/helldivers_adapter.pth


## Exportación y validación cualitativa

Esta celda exporta los mejores resultados como clips MP4 y los muestra dentro del notebook. La validación visual es una parte esencial de este tipo de PoC: una puntuación alta de similitud no basta si el fragmento recuperado no contiene realmente el evento descrito por la consulta.

La exportación añade contexto temporal antes y después de cada ventana mediante `CLIP_CONTEXT_SECONDS`. Esto evita evaluar clips demasiado cortos, donde podría faltar la causa o la consecuencia del evento. Por ejemplo, una explosión puede ocurrir al final de una ventana y la reacción de voz en los segundos siguientes.

Desde una perspectiva de evaluación académica, esta celda sirve como auditoría cualitativa del sistema:

- Permite detectar falsos positivos semánticos.
- Ayuda a comparar recuperación base frente a recuperación con adapter.
- Facilita documentar ejemplos de éxito y fallo en una memoria de tesis.
- Mantiene trazabilidad entre consulta, ranking, puntuación y evidencia audiovisual.

En una fase posterior, estos clips podrían complementarse con métricas cuantitativas, pero para una PoC exploratoria la inspección humana de `Top-K` es una forma eficiente de validar si el pipeline está capturando los momentos esperados.

In [47]:
from IPython.display import Video, display

exported_results = export_result_clips(
    video_path=VIDEO_PATH,
    results=results,
    output_dir=CLIP_OUTPUT_DIR,
    query=QUERY_10,
    context_seconds=CLIP_CONTEXT_SECONDS,
    max_duration_s=duration_s,
)

print_results(exported_results)

for result in exported_results:
    print(f"\nRank {result.rank}: {result.start_s:.2f}s - {result.end_s:.2f}s | score={result.score:.4f}")
    display(Video(str(result.clip_path), embed=False, html_attributes="controls preload='metadata'"))



Top 3 windows:
1. 5080.00s - 5095.00s score=0.2942 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_01_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_5080.00s_5095.00s_clip_5078.00s_5097.00s.mp4
2. 4965.00s - 4980.00s score=0.2647 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_02_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_4965.00s_4980.00s_clip_4963.00s_4982.00s.mp4
3. 5120.00s - 5135.00s score=0.2561 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_03_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_5120.00s_5135.00s_clip_5118.00s_5137.00s.mp4

Rank 1: 5080.00s - 5095.00s | score=0.2942



Rank 2: 4965.00s - 4980.00s | score=0.2647



Rank 3: 5120.00s - 5135.00s | score=0.2561
